In [ ]:
import os
from pathlib import Path

import torch
from diffusers import Kandinsky6SRPipeline

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "dev" / "exports" / "diffusers").exists():
    REPO = REPO.parent
if not (REPO / "dev" / "exports" / "diffusers").exists():
    raise RuntimeError("Run this notebook from the K6 repository or one of its subdirectories.")

In [ ]:
MODEL_REPO = "kandinskylab/Kandinsky-6.0-VSR-distilled2steps-5s-Diffusers"
GPU_ID = 0

pipe = Kandinsky6SRPipeline.from_pretrained(MODEL_REPO, dtype=torch.bfloat16)
pipe.enable_model_cpu_offload(gpu_id=GPU_ID)

In [ ]:
BS = 2
INPUT_VIDEOS = [
    os.path.join(REPO, "assets/vsr_input.mp4")
] * BS
OUTPUT_VIDEOS = [
    os.path.join(REPO, f"outputs/diffusers/k6_vsr/bs{BS}/vsr_x2.25_batchid{i}.mp4")
    for i in range(BS)
]
SEED = 1137
GENERATOR = torch.Generator(device=f"cuda:{GPU_ID}").manual_seed(SEED)

# A video path is decoded and aligned by packaged SR video_io.
# 2.25x means x1.125 pixel pre-upscale followed by the x2 KVAE path.
result = pipe(
    video=INPUT_VIDEOS,
    resolution_scale=2.25,
    save_path=OUTPUT_VIDEOS,
    output_type="pt",
    generator=GENERATOR
)
print(f"frames: {tuple(result.frames.shape)}")
print(f"saved: {result.path}")